# Vol-surface render mock — validate before touching `charts.tsx`

Prototype the front fix on **real** captured data (no production edit). Two real problems found:

1. **Render** — current `VolSurface` uses `mesh3d` over a point cloud with no colorscale → jagged, default blue. Fix: `type:"surface"` over a gridded Z + green→yellow→red colorscale + a **fixed z-axis** so it stops auto-zooming.
2. **Bug** — `orderedBands` (`charts.tsx`) orders the smile axis by **signed delta**, which is *not* monotonic in strike: a 10Δ put (deep OTM, high IV) lands right next to ATM (low IV) → an artificial **spike** in the middle. Fix: order the axis by **log-moneyness** (strike). This affects the 2D `SmileChart` too.

`plotly.py` matches the `plotly.js` schema, so the validated config ports verbatim.

In [1]:
import math, statistics
from pathlib import Path

import plotly.graph_objects as go
from algotrading.frontend.app import create_app
from algotrading.frontend.context import AppContext
from algotrading.infra.storage import ParquetStore
from fastapi.testclient import TestClient

UNDERLYING, TRADE_DATE = "SX5E", "2026-06-12"
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ctx = AppContext(store_root=REPO / "data", configs_dir=REPO / "configs", store=ParquetStore(REPO / "data"))
with TestClient(create_app(ctx)) as c:
    aj = c.get("/api/analytics", params={"underlying": UNDERLYING, "trade_date": TRADE_DATE}).json()
mats = sorted(aj["maturities"], key=lambda m: m["maturity_years"])
print(f"{UNDERLYING} {TRADE_DATE}:", [m["tenor_label"] for m in mats])

/srv/project/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


SX5E 2026-06-12: ['10d', '1m', '3m', '6m', '12m', '18m', '2y', '3y']


In [2]:
# Order bands by MEAN log-moneyness (strike-monotonic) — NOT signed delta. This is the fix for
# the spike. Then build a maturity x band Z-matrix, NaN where a coarse long-dated tenor lacks a
# band (honest holes at the wings).
band_lm = {}
for m in mats:
    for p in m["points"]:
        band_lm.setdefault(p["delta_band"], []).append(p["log_moneyness"])
mean_lm = {b: statistics.mean(v) for b, v in band_lm.items()}
bands = sorted(mean_lm, key=mean_lm.get)
xlm = [mean_lm[b] for b in bands]
ys = [m["maturity_years"] for m in mats]

z = []
for m in mats:
    iv = {p["delta_band"]: p["implied_vol"] for p in m["points"]}
    z.append([iv.get(b, float("nan")) for b in bands])

print("bands (strike order):", bands)
for m, row in zip(mats, z):
    print(f"  {m['tenor_label']:>4}", ["NaN" if isinstance(v, float) and math.isnan(v) else round(v, 3) for v in row])
print("\nEach row should be a smooth monotone skew (puts high -> calls low). No spike.")

bands (strike order): ['10dp', '20dp', '30dp', 'atm', 'atmp', '30dc', '20dc', '10dc']
   10d [0.21, 0.189, 0.176, 0.162, 0.162, 0.154, 0.151, 0.148]
    1m [0.224, 0.196, 0.18, 0.161, 0.161, 0.149, 0.145, 0.142]
    3m [0.248, 0.21, 0.189, 0.167, 0.167, 0.153, 0.147, 0.143]
    6m [0.26, 0.22, 0.198, 0.172, 0.172, 0.156, 0.15, 0.145]
   12m [0.267, 0.226, 0.203, 0.176, 0.176, 0.16, 0.153, 0.147]
   18m ['NaN', 0.225, 0.203, 0.177, 0.177, 0.16, 0.153, 0.147]
    2y ['NaN', 0.23, 0.208, 0.18, 0.18, 0.162, 0.155, 'NaN']
    3y ['NaN', 0.231, 0.209, 0.181, 0.181, 0.163, 0.156, 'NaN']

Each row should be a smooth monotone skew (puts high -> calls low). No spike.


In [3]:
VOL_SCALE = [[0.0, "#1a9850"], [0.25, "#a6d96a"], [0.5, "#fee08b"], [0.75, "#fdae61"], [1.0, "#d73027"]]
Z_RANGE = [0.0, 0.35]  # fixed z-axis (anchored at 0) so the scale is stable across dates


def layout(title):
    return dict(
        title=title,
        scene=dict(
            xaxis=dict(title="log-moneyness"),
            yaxis=dict(title="maturity (y)"),
            zaxis=dict(title="implied vol", range=Z_RANGE),
            aspectratio=dict(x=1.4, y=1.5, z=0.7),
            camera=dict(eye=dict(x=1.8, y=-1.8, z=0.8)),
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=560,
    )

## (A) Current — `mesh3d`, delta-ordered axis (jagged blue + spike)

In [4]:
# Reproduce the current look: delta-ordered x, mesh3d point cloud, default colorscale.
dband = {}
for m in mats:
    for p in m["points"]:
        dband.setdefault(p["delta_band"], p["target_delta"])
delta_order = sorted(dband, key=dband.get)
xs, yy, zz = [], [], []
for m in mats:
    iv = {p["delta_band"]: p["implied_vol"] for p in m["points"]}
    for j, b in enumerate(delta_order):
        if b in iv:
            xs.append(j); yy.append(m["maturity_years"]); zz.append(iv[b])
go.Figure(go.Mesh3d(x=xs, y=yy, z=zz, opacity=0.9)).update_layout(**layout("A — mesh3d, delta-ordered (current)"))

## (C) Proposed — `surface`, log-moneyness axis, green→yellow→red, fixed z, gaps bridged

In [5]:
go.Figure(
    go.Surface(z=z, x=xlm, y=ys, colorscale=VOL_SCALE, colorbar=dict(title="IV"), connectgaps=True)
).update_layout(**layout("C — surface, log-moneyness + fixed z (proposed)"))

## Read-out → port to `charts.tsx`

Reordering by log-moneyness turns every smile into a **monotone textbook skew** — the spike was the delta-ordered axis, a real bug, not the data. To port:

1. **`VolSurface`**: `type:"mesh3d"` → `type:"surface"` with this `z`/`x`(log-moneyness)/`y` grid, `colorscale: VOL_SCALE`, `zaxis.range = [0, 0.35]`, `connectgaps`.
2. **`orderedBands` / `smileAxis`**: order by **log-moneyness**, not `target_delta` — fixes the 2D `SmileChart` spike too.
3. **CSS**: explicit height on `.surface-panel .plot` so the 3D scene stops overlapping the greeks panel.
4. The `svi_a` degeneracy flag does not distort the projected IVs (textbook skew) → over-sensitive, a separate lower-priority finding.